In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text
from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить SQL-запрос и вернуть таблицу pandas."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("Подключились как:", q("SELECT current_user AS login").iloc[0, 0])

## Что определяет динамику числа действующих подписок канала social: приток новых подписок или их прекращение, как на это реагировать?

## Бизнес-вопрос

Как меняется число действующих подписок канала social относительно
других каналов и что сильнее влияет на эту динамику:
приток новых подписок или прекращение действующих?

## Гипотеза

Высокий отток сдерживает рост числа действующих подписок канала
social, а новые подписки не полностью компенсируют прекращения.

In [ ]:
subs = q("""
    SELECT
        subscription_id,
        owner_client_id,
        tariff_id,
        started_at,
        ended_at,
        cancel_reason
    FROM prime.subscriptions
    ORDER BY subscription_id
""")

clients = q("""
    SELECT
        client_id,
        acquisition_channel
    FROM prime.clients
    ORDER BY client_id
""")

print("Подписок:", len(subs))
print("Клиентов:", len(clients))

## 1. Проверка исходных данных

Исходное зерно таблицы `subscriptions` — одна строка на одну подписку.

Пустая дата окончания означает, что подписка
продолжает действовать. Клиентов без указанного канала не удаляем.

In [ ]:
assert len(subs) == 333_905, "Количество подписок не совпало с ожидаемым"
assert len(clients) == 500_000, "Количество клиентов не совпало с ожидаемым"

assert subs["subscription_id"].notna().all(), "Есть подписки без ID"
assert subs["subscription_id"].is_unique, "ID подписок повторяются"

assert clients["client_id"].notna().all(), "Есть клиенты без ID"
assert clients["client_id"].is_unique, "ID клиентов повторяются"

assert subs["owner_client_id"].notna().all(), "Есть подписки без владельца"
assert subs["tariff_id"].notna().all(), "Есть подписки без тарифа"
assert subs["started_at"].notna().all(), "Есть подписки без даты начала"

invalid_dates = subs[
    subs["ended_at"].notna()
    & (subs["ended_at"] < subs["started_at"])
]

audit = pd.DataFrame({
    "проверка": [
        "строк подписок",
        "уникальных ID подписок",
        "строк клиентов",
        "уникальных ID клиентов",
        "подписок без даты окончания",
        "клиентов без канала",
        "окончание раньше начала",
    ],
    "значение": [
        len(subs),
        subs["subscription_id"].nunique(),
        len(clients),
        clients["client_id"].nunique(),
        subs["ended_at"].isna().sum(),
        clients["acquisition_channel"].isna().sum(),
        len(invalid_dates),
    ],
})

display(audit)

print("Типы полей подписок:")
display(subs.dtypes.to_frame("dtype"))

print("Типы полей клиентов:")
display(clients.dtypes.to_frame("dtype"))

## 2. Соединение подписок с каналами привлечения

Канал привлечения хранится в таблице клиентов, поэтому присоединим
его к подписке по владельцу:

`subscriptions.owner_client_id → clients.client_id`.

Ожидаемая кратность: `m:1`.

После соединения: одна строка — одна подписка.

In [ ]:
clients_for_merge = clients.rename(
    columns={
        "client_id": "owner_client_id",
        "acquisition_channel": "channel",
    }
)

subs_channels = subs.merge(
    clients_for_merge,
    on="owner_client_id",
    how="left",
    validate="m:1",
    indicator=True,
)

merge_result = subs_channels["_merge"].value_counts(dropna=False)

display(merge_result.to_frame("строк"))

assert len(subs_channels) == len(subs), (
    "После соединения изменилось количество подписок"
)

assert subs_channels["subscription_id"].is_unique, (
    "После соединения подписки перестали быть уникальными"
)

assert (subs_channels["_merge"] == "both").all(), (
    "Для некоторых владельцев не найден клиент"
)

print("Строк до соединения:", len(subs))
print("Строк после соединения:", len(subs_channels))
print("Уникальных подписок:", subs_channels["subscription_id"].nunique())

In [ ]:
subs_channels["channel"] = (
    subs_channels["channel"]
    .fillna("без канала")
)

subs_channels = subs_channels.drop(columns="_merge")

channel_counts = subs_channels["channel"].value_counts(dropna=False)

display(channel_counts.to_frame("subscriptions"))

assert subs_channels["channel"].isna().sum() == 0, (
    "После обработки остались подписки без категории канала"
)

assert channel_counts.sum() == len(subs_channels), (
    "Не все подписки попали в распределение по каналам"
)

print("Всего подписок:", channel_counts.sum())
print("Все подписки распределены по каналам.")

## 3. Предварительное сравнение оттока по каналам

Сравним отток за май–июль 2026 года. Август не включаем, тк история
заканчивается 30 августа и месяц представлен не полностью.

Отток за месяц это доля подписок, которые были активны на начало
месяца и закончились до начала следующего месяца.

Подписки, оформленные внутри месяца, в знаменатель не включаются.
Одна строка расчётной таблицы это канал × месяц.

In [ ]:
month_starts = pd.date_range(
    start="2026-05-01",
    end="2026-07-01",
    freq="MS",
    tz="UTC",
)

monthly_parts = []

for month_start in month_starts:
    next_month = month_start + pd.offsets.MonthBegin(1)

    active_at_start = (
        (subs_channels["started_at"] < month_start)
        & (
            subs_channels["ended_at"].isna()
            | (subs_channels["ended_at"] >= month_start)
        )
    )

    ended_in_month = (
        active_at_start
        & subs_channels["ended_at"].notna()
        & (subs_channels["ended_at"] < next_month)
    )

    month_data = subs_channels.assign(
        active_at_start=active_at_start,
        ended_in_month=ended_in_month,
    )

    month_summary = (
        month_data
        .groupby("channel", as_index=False)
        .agg(
            active_at_start=("active_at_start", "sum"),
            ended=("ended_in_month", "sum"),
        )
    )

    month_summary["month"] = month_start.strftime("%Y-%m")
    monthly_parts.append(month_summary)

churn_by_channel = pd.concat(
    monthly_parts,
    ignore_index=True,
)

churn_by_channel["churn_pct"] = (
    churn_by_channel["ended"]
    / churn_by_channel["active_at_start"]
    * 100
)

assert (
    churn_by_channel["ended"]
    <= churn_by_channel["active_at_start"]
).all(), "Прекращений оказалось больше начальной базы"

churn_table = churn_by_channel.pivot(
    index="channel",
    columns="month",
    values="churn_pct",
)

churn_table.round(2).sort_values(
    "2026-07",
    ascending=False,
)

### Предварительный вывод

Канал social имеет самый высокий отток во всех трёх рассматриваемых
месяцах: от 6,82% до 7,22%. У performance показатель
составляет 5,39–5,52%, у organic — 2,14–2,26%.

Высокий отток social является устойчивым наблюдением,
а не результатом одного аномального месяца. Но из этого расчёта
 не следует, что число действующих подписок social растёт медленнее:
высокий отток может компенсироваться большим количеством новых подписок.

## 4. Баланс действующих подписок

Проверим, компенсируется ли высокий отток social притоком новых
подписок. Для каждого канала посчитаем: действующие подписки на 1 мая; новые подписки с 1 мая по 31 июля; все прекращения за тот же период; действующие подписки на 1 августа.

Одна строка итоговой таблицы это канал. 

In [ ]:
period_start = pd.Timestamp("2026-05-01", tz="UTC")
period_end = pd.Timestamp("2026-08-01", tz="UTC")

balance_source = subs_channels.assign(
    active_at_start=(
        (subs_channels["started_at"] < period_start)
        & (
            subs_channels["ended_at"].isna()
            | (subs_channels["ended_at"] >= period_start)
        )
    ),
    new_subscription=(
        (subs_channels["started_at"] >= period_start)
        & (subs_channels["started_at"] < period_end)
    ),
    ended_subscription=(
        subs_channels["ended_at"].notna()
        & (subs_channels["ended_at"] >= period_start)
        & (subs_channels["ended_at"] < period_end)
    ),
    active_at_end=(
        (subs_channels["started_at"] < period_end)
        & (
            subs_channels["ended_at"].isna()
            | (subs_channels["ended_at"] >= period_end)
        )
    ),
)

base_balance = (
    balance_source
    .groupby("channel", as_index=False)
    .agg(
        base_start=("active_at_start", "sum"),
        new_subscriptions=("new_subscription", "sum"),
        ended_subscriptions=("ended_subscription", "sum"),
        base_end=("active_at_end", "sum"),
    )
)

base_balance["net_growth"] = (
    base_balance["base_end"]
    - base_balance["base_start"]
)

base_balance["growth_from_flows"] = (
    base_balance["new_subscriptions"]
    - base_balance["ended_subscriptions"]
)

base_balance["balance_error"] = (
    base_balance["net_growth"]
    - base_balance["growth_from_flows"]
)

assert base_balance["balance_error"].eq(0).all(), (
    "Баланс подписок не сошёлся"
)

base_balance["growth_pct"] = (
    base_balance["net_growth"]
    / base_balance["base_start"]
    * 100
)

base_balance.sort_values(
    "growth_pct",
    ascending=False,
).round(2)

### Результат проверки баланса

За май–июль число действующих подписок social выросло с 20 749
до 21 626 (на 877 подписок,на 4,23%). Это самый низкий темп
роста среди каналов: у остальных он составил от 5,25% до 8,28%.

За период канал получил 5 330 новых подписок и потерял 4 453.
Следовательно, приток превысил прекращения, и база не сократилась.
Первоначальная гипотеза подтвердилась частично: проблема заключается
не в отсутствии роста, а в том, что прекращения поглощают большую
часть притока и замедляют рост в сравнении с  другими каналами.

In [ ]:
flow_rates = base_balance.copy()

flow_rates["new_rate_pct"] = (
    flow_rates["new_subscriptions"]
    / flow_rates["base_start"]
    * 100
)

flow_rates["ended_rate_pct"] = (
    flow_rates["ended_subscriptions"]
    / flow_rates["base_start"]
    * 100
)

flow_rates["net_rate_pct"] = (
    flow_rates["net_growth"]
    / flow_rates["base_start"]
    * 100
)

flow_rates["rate_check"] = (
    flow_rates["net_rate_pct"]
    - (
        flow_rates["new_rate_pct"]
        - flow_rates["ended_rate_pct"]
    )
)

assert flow_rates["rate_check"].abs().lt(1e-10).all(), (
    "Темпы притока и прекращения не сходятся с чистым ростом"
)

flow_rates[
    [
        "channel",
        "new_rate_pct",
        "ended_rate_pct",
        "net_rate_pct",
    ]
].sort_values(
    "net_rate_pct",
    ascending=False,
).round(2)

### Что сдерживает рост social

У social самый высокий относительный приток новых подписок:
25,69% от базы на начало периода. Медленный рост
нельзя объяснить недостатком новых подписок.

Одновременно у social самый высокий поток прекращений: 21,46%
от начальной базы. После его вычета чистый рост составляет только
4,23%, является минимальным результатом среди каналов.

Таким образом, основным ограничением роста social является удержание,
а не привлечение. Гипотеза о высоком оттоке подтвердилась, а
предположение о недостаточном притоке новых подписок не подтвердилась.

## 5. Проверка влияния тарифного состава

Высокий отток social может быть связан с составом его подписок:
например, с большей долей тарифа, который в целом прекращают чаще.

Чтобы проверить это объяснение, рассчитаем отток отдельно внутри
каждого тарифа. Используем те же месяцы и то же определение оттока.

In [ ]:
tariff_monthly_parts = []

for month_start in month_starts:
    next_month = month_start + pd.offsets.MonthBegin(1)

    active_at_start = (
        (subs_channels["started_at"] < month_start)
        & (
            subs_channels["ended_at"].isna()
            | (subs_channels["ended_at"] >= month_start)
        )
    )

    month_data = subs_channels.loc[
        active_at_start,
        ["channel", "tariff_id", "ended_at"],
    ].copy()

    month_data["ended_in_month"] = (
        month_data["ended_at"].notna()
        & (month_data["ended_at"] < next_month)
    )

    month_summary = (
        month_data
        .groupby(
            ["channel", "tariff_id"],
            as_index=False,
        )
        .agg(
            active_months=("ended_in_month", "size"),
            ended=("ended_in_month", "sum"),
        )
    )

    tariff_monthly_parts.append(month_summary)

churn_by_tariff = (
    pd.concat(
        tariff_monthly_parts,
        ignore_index=True,
    )
    .groupby(
        ["channel", "tariff_id"],
        as_index=False,
    )
    .agg(
        active_months=("active_months", "sum"),
        ended=("ended", "sum"),
    )
)

churn_by_tariff["churn_pct"] = (
    churn_by_tariff["ended"]
    / churn_by_tariff["active_months"]
    * 100
)

assert (
    churn_by_tariff["ended"]
    <= churn_by_tariff["active_months"]
).all(), "Прекращений больше числа активных подписок-месяцев"

tariff_table = churn_by_tariff.pivot(
    index="channel",
    columns="tariff_id",
    values="churn_pct",
)

tariff_table.round(2).sort_values(
    1,
    ascending=False,
)

### Вывод по тарифам

У social самый высокий отток внутри обоих тарифов: 7,17% по тарифу 1
и 6,85% по тарифу 2. У ближайшего канала performance соответствующие
значения составляют 5,47% и 5,44%.

Следовательно, высокий общий отток social не объясняется только
тарифным составом подписок. Различие сохраняется при сравнении
подписок одного тарифа.

## 6. Проверка влияния возраста подписки

Отток social может быть выше из-за состава базы по возрасту:
допустим, если в канале больше недавно оформленных подписок.

Разделим подписки по возрасту на начало каждого месяца и сравним
отток каналов внутри одинаковых возрастных групп: менее 30 дней; от 30 до 89 дней; от 90 до 179 дней; 180 дней и более.

Единица подсчёта — один месяц активности подписки.

In [ ]:
age_monthly_parts = []

age_labels = [
    "1. Менее 30 дней",
    "2. От 30 до 89 дней",
    "3. От 90 до 179 дней",
    "4. От 180 дней",
]

for month_start in month_starts:
    next_month = month_start + pd.offsets.MonthBegin(1)

    active_at_start = (
        (subs_channels["started_at"] < month_start)
        & (
            subs_channels["ended_at"].isna()
            | (subs_channels["ended_at"] >= month_start)
        )
    )

    month_data = subs_channels.loc[
        active_at_start,
        ["channel", "started_at", "ended_at"],
    ].copy()

    month_data["age_days"] = (
        month_start - month_data["started_at"]
    ).dt.days

    month_data["age_group"] = pd.cut(
        month_data["age_days"],
        bins=[0, 30, 90, 180, float("inf")],
        labels=age_labels,
        right=False,
    )

    month_data["ended_in_month"] = (
        month_data["ended_at"].notna()
        & (month_data["ended_at"] < next_month)
    )

    month_summary = (
        month_data
        .groupby(
            ["channel", "age_group"],
            as_index=False,
            observed=True,
        )
        .agg(
            active_months=("ended_in_month", "size"),
            ended=("ended_in_month", "sum"),
        )
    )

    age_monthly_parts.append(month_summary)

churn_by_age = (
    pd.concat(
        age_monthly_parts,
        ignore_index=True,
    )
    .groupby(
        ["channel", "age_group"],
        as_index=False,
        observed=True,
    )
    .agg(
        active_months=("active_months", "sum"),
        ended=("ended", "sum"),
    )
)

churn_by_age["churn_pct"] = (
    churn_by_age["ended"]
    / churn_by_age["active_months"]
    * 100
)

assert (
    churn_by_age["ended"]
    <= churn_by_age["active_months"]
).all(), "Прекращений больше числа активных подписок-месяцев"

age_table = churn_by_age.pivot(
    index="channel",
    columns="age_group",
    values="churn_pct",
)

age_table.round(2)

### Вывод по возрасту подписок

У social самый высокий отток во всех возрастных группах. Среди
подписок младше 30 дней он составляет 7,13%, среди подписок
старше 180 дней составляет 6,93%. Но performance показатели
составляют 5,35% и 5,48%, а у organic — 2,19% и 2,22%.

Значит, высокий отток social не объясняется только возрастным
составом его базы. Различие сохраняется как среди новых, так и среди
давно действующих подписок.

## 7. Причины прекращения подписок

Рассмотрим все подписки, закончившиеся с 1 мая по 31 июля.
В отличие от расчёта оттока, здесь учитываются также подписки,
которые начались и закончились внутри периода.
Используем два показателя:
1. доля причины среди всех прекращений канала (описывает структуру прекращений);
2. число прекращений с этой причиной относительно размера базы
   канала на начало периода(влияние на базу).

In [ ]:
ended_period = subs_channels.loc[
    subs_channels["ended_at"].notna()
    & (subs_channels["ended_at"] >= period_start)
    & (subs_channels["ended_at"] < period_end),
    ["channel", "cancel_reason"],
].copy()

ended_period["reason"] = (
    ended_period["cancel_reason"]
    .fillna("не указана")
)

reason_counts = (
    ended_period
    .groupby(
        ["channel", "reason"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "n_cancellations"})
)

reason_counts["share_pct"] = (
    reason_counts["n_cancellations"]
    / reason_counts.groupby("channel")["n_cancellations"].transform("sum")
    * 100
)

reason_counts = reason_counts.merge(
    base_balance[
        ["channel", "base_start", "ended_subscriptions"]
    ],
    on="channel",
    how="left",
    validate="m:1",
)

reason_counts["rate_to_base_pct"] = (
    reason_counts["n_cancellations"]
    / reason_counts["base_start"]
    * 100
)

reason_control = (
    reason_counts
    .groupby("channel", as_index=False)
    .agg(
        counted_cancellations=("n_cancellations", "sum"),
        expected_cancellations=("ended_subscriptions", "first"),
    )
)

assert (
    reason_control["counted_cancellations"]
    == reason_control["expected_cancellations"]
).all(), "Причины не сходятся с общим числом прекращений"

print("Доли причин среди прекращений, %:")

reason_share_table = reason_counts.pivot(
    index="reason",
    columns="channel",
    values="share_pct",
)

display(reason_share_table.round(2))

print("Прекращения относительно начальной базы, %:")

reason_rate_table = reason_counts.pivot(
    index="reason",
    columns="channel",
    values="rate_to_base_pct",
)

display(reason_rate_table.round(2))

### Вывод по причинам прекращения

Структура причин прекращения у social похожа на другие каналы.
Наиболее распространённая причина — `not_using`: 33,42% прекращений
social. Далее следуют `too_expensive` — 23,87%, `switched` — 18,95%
и `payment_failed` — 13,45%.

Высокий отток social нельзя объяснить одной особой
причиной: доли причин близки к другим каналам. Относительно
начальной базы потери social выше по каждой крупной причине.
Наибольший вклад дают `not_using` — 7,17% базы, `too_expensive` —
5,12% и `switched` — 4,07%.

## 8. Оценка масштаба возможного улучшения

Сравним social со средним уровнем остальных известных каналов.
Для каждой причины рассчитаем частоту прекращений относительно
начальной базы остальных каналов и применим её к начальной базе social.

Разница между фактическим и расчётным числом прекращений показывает
масштаб возможного улучшения при достижении среднего уровня остальных
каналов. Это ориентир для приоритизации.

In [ ]:
benchmark_channels = base_balance.loc[
    ~base_balance["channel"].isin(
        ["social", "без канала"]
    ),
    "channel",
]

benchmark_base = base_balance.loc[
    base_balance["channel"].isin(benchmark_channels),
    "base_start",
].sum()

benchmark_reasons = (
    reason_counts.loc[
        reason_counts["channel"].isin(benchmark_channels)
    ]
    .groupby("reason", as_index=False)
    .agg(
        benchmark_cancellations=("n_cancellations", "sum")
    )
)

benchmark_reasons["benchmark_rate_pct"] = (
    benchmark_reasons["benchmark_cancellations"]
    / benchmark_base
    * 100
)

social_base = base_balance.loc[
    base_balance["channel"] == "social",
    "base_start",
].iloc[0]

social_reasons = reason_counts.loc[
    reason_counts["channel"] == "social",
    ["reason", "n_cancellations"],
].rename(
    columns={"n_cancellations": "actual_cancellations"}
)

opportunity = social_reasons.merge(
    benchmark_reasons[
        ["reason", "benchmark_rate_pct"]
    ],
    on="reason",
    how="left",
    validate="1:1",
)

opportunity["expected_at_benchmark"] = (
    social_base
    * opportunity["benchmark_rate_pct"]
    / 100
)

opportunity["excess_cancellations"] = (
    opportunity["actual_cancellations"]
    - opportunity["expected_at_benchmark"]
)

opportunity = opportunity.sort_values(
    "excess_cancellations",
    ascending=False,
)

display(
    opportunity.round({
        "benchmark_rate_pct": 2,
        "expected_at_benchmark": 0,
        "excess_cancellations": 0,
    })
)

print(
    "Потенциально сохраняемые подписки:",
    round(opportunity["excess_cancellations"].sum()),
)

### Масштаб возможного улучшения

Фактическое число прекращений social за период превышает расчётный
уровень остальных известных каналов примерно на 2 218 подписок.

Наибольший вклад в разрыв дают причины `not_using` — 729 подписок,
`too_expensive` — 524 и `switched` — 447. На них приходится основная
часть потенциального улучшения.


## 9. Факторная декомпозиция изменения базы social

Используем:

**база на конец = база на начало + новые подписки − прекращения.**

Новые подписки дают положительный вклад в изменение базы,
прекращения — отрицательный. Сумма вкладов должна точно совпасть
с фактическим изменением числа действующих подписок.

In [ ]:
social_balance = (
    base_balance
    .set_index("channel")
    .loc["social"]
)

social_start = int(social_balance["base_start"])
social_new = int(social_balance["new_subscriptions"])
social_ended = int(social_balance["ended_subscriptions"])
social_end = int(social_balance["base_end"])

assert (
    social_start + social_new - social_ended
    == social_end
), "Факторная декомпозиция не сошлась"

decomposition = pd.DataFrame({
    "элемент": [
        "База на 1 мая",
        "Новые подписки",
        "Прекращения",
        "База на 1 августа",
    ],
    "значение": [
        social_start,
        social_new,
        -social_ended,
        social_end,
    ],
})

display(decomposition)

potential_retained = int(
    round(opportunity["excess_cancellations"].sum())
)

scenario_end = social_end + potential_retained

scenario_growth_pct = (
    scenario_end / social_start - 1
) * 100

print("Фактический рост базы:", f"{social_balance['growth_pct']:.2f}%")
print("Сценарная база на 1 августа:", scenario_end)
print("Сценарный рост базы:", f"{scenario_growth_pct:.2f}%")

### Вывод факторной декомпозиции

За май–июль канал social привлёк 5 330 новых подписок, но потерял
4 453. Поэтому из потенциального прироста сохранилось только
877 подписок: база выросла с 20 749 до 21 626 (на 4,23%).

При снижении частоты прекращений до среднего уровня остальных
известных каналов расчётная база могла бы составить около 23 844
подписок, рост — около 14,92% (сценарная оценка масштаба
возможного улучшения).

## 10. Визуализация результатов

### Рост числа действующих подписок по каналам

Сравним чистый рост базы каждого канала относительно количества
действующих подписок на 1 мая 2026 года.

In [ ]:
growth_plot = (
    flow_rates[
        ["channel", "net_rate_pct"]
    ]
    .sort_values("net_rate_pct")
)

colors = [
    "#C1272D" if channel == "social" else "#B8C0CC"
    for channel in growth_plot["channel"]
]

fig, ax = plt.subplots(
    figsize=(9, 5),
    layout="constrained",
)

bars = ax.barh(
    growth_plot["channel"],
    growth_plot["net_rate_pct"],
    color=colors,
    zorder=2,
)

for bar, value in zip(bars, growth_plot["net_rate_pct"]):
    ax.text(
        value + 0.08,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}%".replace(".", ","),
        va="center",
    )

ax.set_title(
    "База social выросла на 4,23% — меньше остальных каналов",
    loc="left",
)

ax.set_xlabel(
    "Рост действующих подписок за май–июль 2026 года, %"
)
ax.set_ylabel("")

ax.set_xlim(
    0,
    growth_plot["net_rate_pct"].max() * 1.18,
)

ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", color="#E5E7EB", zorder=0)
ax.set_axisbelow(True)

plt.show()

### Устойчивость оттока social

Сравним ежемесячный отток social со средним уровнем остальных
известных каналов. Средний показатель рассчитываем как отношение
общего числа прекращений к общей начальной базе этих каналов.

In [ ]:
other_monthly = (
    churn_by_channel.loc[
        ~churn_by_channel["channel"].isin(
            ["social", "без канала"]
        )
    ]
    .groupby("month", as_index=False)
    .agg(
        active_at_start=("active_at_start", "sum"),
        ended=("ended", "sum"),
    )
)

other_monthly["other_churn_pct"] = (
    other_monthly["ended"]
    / other_monthly["active_at_start"]
    * 100
)

social_monthly = churn_by_channel.loc[
    churn_by_channel["channel"] == "social",
    ["month", "churn_pct"],
].rename(
    columns={"churn_pct": "social_churn_pct"}
)

churn_comparison = social_monthly.merge(
    other_monthly[
        ["month", "other_churn_pct"]
    ],
    on="month",
    how="left",
    validate="1:1",
)

churn_comparison["month_label"] = churn_comparison["month"].map({
    "2026-05": "Май",
    "2026-06": "Июнь",
    "2026-07": "Июль",
})

churn_comparison

fig, ax = plt.subplots(
    figsize=(9, 5),
    layout="constrained",
)

x = range(len(churn_comparison))

ax.plot(
    x,
    churn_comparison["other_churn_pct"],
    color="#9CA3AF",
    marker="o",
    linewidth=2,
    label="Остальные известные каналы",
)

ax.plot(
    x,
    churn_comparison["social_churn_pct"],
    color="#C1272D",
    marker="o",
    linewidth=3,
    label="social",
)

for position, value in zip(
    x,
    churn_comparison["social_churn_pct"],
):
    ax.annotate(
        f"{value:.2f}%".replace(".", ","),
        (position, value),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        color="#C1272D",
    )

for position, value in zip(
    x,
    churn_comparison["other_churn_pct"],
):
    ax.annotate(
        f"{value:.2f}%".replace(".", ","),
        (position, value),
        xytext=(0, -16),
        textcoords="offset points",
        ha="center",
        color="#6B7280",
    )

ax.set_title(
    "Отток social держится около 7% — вдвое выше остальных каналов",
    loc="left",
)

ax.set_xticks(
    list(x),
    churn_comparison["month_label"],
)

ax.set_ylabel("Отток за месяц, %")
ax.set_xlabel("2026 год")
ax.set_ylim(0, 8.5)

ax.legend(
    frameon=False,
    loc="upper right",
)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
ax.grid(axis="x", visible=False)

plt.show()

### Факторная декомпозиция базы social

Каскадная диаграмма показывает переход от базы на 1 мая к базе
на 1 августа. Приток новых подписок увеличивает базу, прекращения
уменьшают её.

In [ ]:
waterfall = pd.DataFrame({
    "label": [
        "База\nна 1 мая",
        "Новые\nподписки",
        "Прекращения",
        "База\nна 1 августа",
    ],
    "height": [
        social_start,
        social_new,
        -social_ended,
        social_end,
    ],
    "bottom": [
        0,
        social_start,
        social_start + social_new,
        0,
    ],
})

assert (
    social_start + social_new - social_ended
    == social_end
), "Каскад не сходится с итоговой базой"

colors = [
    "#6B7280",
    "#2A9D8F",
    "#C1272D",
    "#6B7280",
]

labels = [
    f"{social_start:,}".replace(",", " "),
    f"+{social_new:,}".replace(",", " "),
    f"−{social_ended:,}".replace(",", " "),
    f"{social_end:,}".replace(",", " "),
]

fig, ax = plt.subplots(
    figsize=(9, 5),
    layout="constrained",
)

bar_width = 0.65
positions = range(len(waterfall))

bars = ax.bar(
    positions,
    waterfall["height"] / 1000,
    bottom=waterfall["bottom"] / 1000,
    width=bar_width,
    color=colors,
    zorder=2,
)

# Соединительные линии между ступенями каскада.
connector_levels = [
    social_start,
    social_start + social_new,
    social_end,
]

for position, level in enumerate(connector_levels):
    ax.plot(
        [
            position + bar_width / 2,
            position + 1 - bar_width / 2,
        ],
        [level / 1000, level / 1000],
        color="#9CA3AF",
        linewidth=1,
        zorder=1,
    )

ax.bar_label(
    bars,
    labels=labels,
    padding=4,
)

share_consumed = (
    social_ended / social_new * 100
)

ax.set_title(
    f"Прекращения поглотили {share_consumed:.0f}% притока social",
    loc="left",
)

ax.set_xticks(
    list(positions),
    waterfall["label"],
)

ax.set_ylabel("тыс. действующих подписок")
ax.set_xlabel("Период: 1 мая — 1 августа 2026 года, UTC")

ax.set_ylim(
    0,
    (social_start + social_new) / 1000 * 1.12,
)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB", zorder=0)
ax.grid(axis="x", visible=False)
ax.set_axisbelow(True)

plt.show()

### Причины избыточных прекращений social

Покажем, сколько прекращений по каждой причине превышает расчётный
уровень остальных известных каналов. Значения являются сценарной
оценкой и используются для определения приоритетов.

In [ ]:
reason_names = {
    "not_using": "Не пользуются",
    "too_expensive": "Слишком дорого",
    "switched": "Перешли на другой сервис",
    "payment_failed": "Не прошла оплата",
    "other": "Другая причина",
}

opportunity_plot = opportunity.copy()

opportunity_plot["reason_label"] = (
    opportunity_plot["reason"]
    .map(reason_names)
    .fillna(opportunity_plot["reason"])
)

opportunity_plot = opportunity_plot.sort_values(
    "excess_cancellations",
    ascending=False,
)

top_reason = opportunity_plot.iloc[0]
top_excess = int(round(top_reason["excess_cancellations"]))
total_excess = int(
    round(opportunity_plot["excess_cancellations"].sum())
)

colors = [
    "#C1272D" if reason == top_reason["reason"] else "#B8C0CC"
    for reason in opportunity_plot["reason"]
]

fig, ax = plt.subplots(
    figsize=(9, 5),
    layout="constrained",
)

bars = ax.barh(
    opportunity_plot["reason_label"],
    opportunity_plot["excess_cancellations"],
    color=colors,
    zorder=2,
)

ax.invert_yaxis()

for bar, value in zip(
    bars,
    opportunity_plot["excess_cancellations"],
):
    ax.text(
        value + 12,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.0f}",
        va="center",
    )

title = (
    f"Неиспользование — {top_excess:,} из "
    f"{total_excess:,} избыточных прекращений"
).replace(",", " ")

ax.set_title(
    title,
    loc="left",
)

ax.set_xlabel(
    "Прекращения сверх среднего уровня остальных каналов"
)
ax.set_ylabel("")

ax.set_xlim(
    0,
    opportunity_plot["excess_cancellations"].max() * 1.18,
)

ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(
    axis="x",
    color="#E5E7EB",
    zorder=0,
)
ax.grid(
    axis="y",
    visible=False,
)
ax.set_axisbelow(True)

plt.show()

## 11. Итоговые проверки

Проверим, что основные результаты
согласованы между собой и контрольные равенства выполняются.

In [ ]:
final_checks = pd.Series({
    "зерно подписок сохранено":
        len(subs_channels) == subs["subscription_id"].nunique(),

    "баланс базы сошёлся":
        base_balance["balance_error"].eq(0).all(),

    "декомпозиция social сошлась":
        social_start + social_new - social_ended == social_end,

    "social имеет минимальный рост":
        flow_rates.set_index("channel")["net_rate_pct"].idxmin()
        == "social",

    "social имеет максимальный отток в каждом месяце":
        churn_table.idxmax().eq("social").all(),

    "social имеет максимальный отток в каждом тарифе":
        tariff_table.idxmax().eq("social").all(),

    "social имеет максимальный отток в каждой возрастной группе":
        age_table.idxmax().eq("social").all(),

    "причины сходятся с числом прекращений":
        (
            reason_control["counted_cancellations"]
            == reason_control["expected_cancellations"]
        ).all(),
})

display(final_checks.to_frame("проверка пройдена"))

assert final_checks.all(), "Одна или несколько итоговых проверок не пройдены"

print("Все итоговые проверки пройдены.")

## 12. Вывод и рекомендации

### Что случилось?

С 1 мая по 1 августа 2026 года число действующих подписок social
выросло с 20 749 до 21 626 — на 877 подписок, или на 4,23%.
Это минимальный темп роста среди каналов: у остальных он составил
от 5,25% до 8,28%.

### Из-за чего случилось?

За период канал получил 5 330 новых подписок и потерял 4 453.
Таким образом, прекращения поглотили около 84% притока.

Проблема не связана с недостатком новых подписок: относительный
приток social — 25,69% начальной базы — является самым высоким
среди каналов. Рост сдерживает поток прекращений, равный 21,46%
начальной базы.

Высокий отток social сохраняется во всех трёх месяцах, внутри обоих
тарифов и во всех возрастных группах подписок. Поэтому различие
нельзя объяснить только тарифным или возрастным составом.

### Что показали причины прекращения?

Структура причин social похожа на другие каналы, поэтому одной
специфической причины не обнаружено. Наибольший разрыв относительно
среднего уровня остальных известных каналов связан с причинами
`not_using` — 729 подписок, `too_expensive` — 524 и `switched` —
447.

При снижении частоты прекращений до среднего уровня остальных
известных каналов расчётная база social могла бы быть примерно
на 2 218 подписок больше. Это сценарная оценка, а не прогноз.

### Рекомендации

1. Командам продукта и CRM провести контролируемый пилот удержания
   для новых подписчиков social: улучшить знакомство с возможностями
   подписки и напоминать о неиспользуемых преимуществах в первые
   недели после оформления через социальные сети (откуда и пришли клиенты).
2. Для клиентов с риском причин `too_expensive` и `switched`
   протестировать сообщения о ценности подписки и персональные
   предложения. Не применять скидку ко всему каналу без эксперимента.
3. Для `payment_failed` проверить сценарий повторного списания,
   напоминания об оплате и обновления платёжного способа.
4. Оценивать пилот по оттоку среди подписок, активных на начало
   периода, и по чистому росту базы. Сравнивать тестовую и контрольную
   группы внутри одинаковых тарифов и возрастных групп.

### Ограничения

Анализ выявляет устойчивую связь между каналом social и повышенным
оттоком, но не доказывает причинное влияние канала. Причины прекращения
могут быть неполными или субъективными. Исследование охватывает три
месяца, сценарий 2 218 сохранённых подписок предполагает, что
средний уровень остальных каналов достижим для social.

Единицей анализа является подписка владельца. Участники шеринга
не считаются отдельными подписками.